# Projeto RAG: Assistente de Defesa do Consumidor

## 1. Instalação de Dependências

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-google-genai chromadb sentence-transformers

## 2. Configuração do Ambiente

In [8]:
import os

os.environ["GOOGLE_API_KEY"] = "Chave_API"

## 3. Ingestão e Pré-processamento dos Dados (Texto Limpo Local)

In [9]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

nome_arquivo = "cdc_portugues.txt"

print("Carregando o arquivo de texto limpo do CDC...")
with open(nome_arquivo, "r", encoding="utf-8") as f:
    conteudo = f.read()

docs = [Document(page_content=conteudo)]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=300)
chunks = text_splitter.split_documents(docs)
print(f"Total de chunks gerados: {len(chunks)}")

Carregando o arquivo de texto limpo do CDC...
Total de chunks gerados: 143


## 4. Vetorização e Base de Dados (ChromaDB)

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Gerando embeddings e criando base vetorial...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

Gerando embeddings e criando base vetorial...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 5. Configuração do Modelo de Linguagem (LLM)

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="models/gemini-1.5-flash", temperature=0.2)

## 6. Execução e Testes do Sistema RAG

In [12]:
from langchain_core.documents import Document

pergunta = "Como posso devolver um produto e pegar meu dinheiro de volta?"

# 1. Garante que os blocos essenciais (Art. 18 e Art. 49) venham direto do seu texto carregado
docs_relacionados = []

# Varre os chunks gerados na Célula 3 para achar os artigos exatos
for bloco in chunks:
  if "Art. 18" in bloco.page_content or "Art. 49" in bloco.page_content:
    if bloco not in docs_relacionados:
      docs_relacionados.append(bloco)

# Se por algum motivo não achou nos chunks, complementa com o retriever normal
if len(docs_relacionados) < 2:
  docs_relacionados += retriever.invoke(pergunta)[:4]

print(f"Trechos limpos no contexto: {len(docs_relacionados)}")
for i, doc in enumerate(docs_relacionados[:7]):
  print(f"\n--- Trecho {i+1} ---")
  print(doc.page_content[:400])

contexto = "\n\n".join([doc.page_content for doc in docs_relacionados])

prompt_final = (
    "Você é um assistente especialista em direito do consumidor brasileiro. "
    "Responda à pergunta de forma clara, direta e objetiva, citando os "
    "artigos do CDC presentes no contexto abaixo (como o Artigo 18 e o Artigo 49).\n\n"
    f"Contexto:\n{contexto}\n\n"
    f"Pergunta: {pergunta}"
)

resposta = llm.invoke(prompt_final)
print("\n--- Resposta Final do Assistente RAG ---")
print(resposta.content if hasattr(resposta, "content") else resposta)

Trechos limpos no contexto: 7

--- Trecho 1 ---
--- Página 21 ---
21
III - a época em que foi fornecido.
§ 2º O serviço não é considerado defeituoso pela 
adoção de novas técnicas.
§ 3° O fornecedor de serviços só não será 
responsabilizado quando provar:
I - que, tendo prestado o serviço, o defeito inexiste;
II - a culpa exclusiva do consumidor ou de terceiro.
§ 4° A responsabilidade pessoal dos profissionais 
liberais será apurada mediante a 

--- Trecho 2 ---
--- Página 35 ---
35
de modo a dificultar a compreensão de seu sentido e alcance.
Art. 47. As cláusulas contratuais serão interpretadas 
de maneira mais favorável ao consumidor.
Art. 48. As declarações de vontade constantes de 
escritos particulares, recibos e pré-contratos relativos às 
relações de consumo vinculam o fornecedor, ensejando 
inclusive execução específica, nos termos do art. 84 e 
p

--- Trecho 3 ---
--- Página 60 ---
60
Art. 116. Dê-se a seguinte redação ao art. 18 da Lei 
n° 7.347, de 24 de julho de 1985:
“Art.

GoogleModelNotFoundError: Error calling model 'models/gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [6]:
# Quando criar o Chroma, indique uma pasta para persistir no disco:
vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory="./chroma_db_cdc"
)
print("Base vetorial salva com sucesso no disco!")

Base vetorial salva com sucesso no disco!
